In [128]:
!git clone https://github.com/zinaelnahel/NLP-automated-customer-reviews.git
%cd NLP-automated-customer-reviews

Cloning into 'NLP-automated-customer-reviews'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 28 (delta 4), reused 23 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 3.54 MiB | 3.93 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews/NLP-automated-customer-reviews


In [129]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/1429_1.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/1429_1.csv")

df = pd.read_csv(DATA_PATH, low_memory=False)
display(df.head())

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [130]:
category_counts = (
    df["categories"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

display(category_counts)

,count
categories,
Electronics,42291
Computers & Tablets,21719
Tablets,21383
All Tablets,18413
iPad & Tablets,17784
...,...
Lotions,1
Rice Dishes,1
Ready Meals,1


In [131]:
df["id"].nunique()

42

## Goal and approach

Group distinct product metadata profiles into 4–6 product meta-categories using category paths, product names and brands, and review language. Review titles and bodies are embedded individually and averaged per product profile, so each profile contributes one row. Category, product, and review embedding blocks are weighted 4, 2, and 1; ratings and recommendation fields are not model inputs.

We compare k=4, 5, and 6 with cosine silhouette and inspect product and review examples. The current selection uses k=4 as a simpler, interpretable grouping; k=6 has a slightly higher silhouette, so the selection is qualitative rather than a claim that four is objectively optimal. Cluster numbers are arbitrary; the labels below interpret each run's summaries and can change if the data or model changes.

In [132]:
print(f"Rows: {len(df):,}; columns: {df.shape[1]}")
print(f"Products with names: {df['name'].nunique(dropna=True):,}")
print(f"Unique product IDs: {df['id'].nunique(dropna=True):,}")
display(df[['name', 'brand', 'categories', 'reviews.title', 'reviews.text']].isna().sum().rename('missing').to_frame())

display(df['name'].fillna('[name missing]').value_counts().head(10).rename_axis('product').to_frame('review_count'))

Rows: 34,660; columns: 21
Products with names: 48
Unique product IDs: 42


,missing
name,6760
brand,0
categories,0
reviews.title,6
reviews.text,1


,review_count
product,
"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta",10966
[name missing],6760
"Echo (White),,,\nEcho (White),,,",3309
"Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrome Paperwhite - touchscreen - Wi-Fi - black,,,",3176
"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta",2814
"Amazon Fire Tv,,,\nAmazon Fire Tv,,,",2527
"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case",1685
"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,",1038
"Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi) with Adaptive Built-in Light, PagePress Sensors, Wi-Fi - Includes Special Offers,",580


## Build product and review features

The clustering unit is one distinct product metadata profile, not one review. Category paths, product name/brand fields, and each product profile's aggregated review language use sentence-transformer embeddings. Review embeddings are averaged per profile, so every profile contributes one row regardless of review count. Category, product, and review blocks are weighted 4, 2, and 1 respectively. Ratings and recommendation fields are excluded.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

product_columns = ['id', 'name', 'brand', 'categories']
product_df = df[product_columns].drop_duplicates().reset_index(drop=True)

def unique_category_labels(value):
    seen = set()
    terms = []
    for term in str(value).split(','):
        term = term.strip()
        normalized = term.casefold()
        if term and normalized not in seen:
            seen.add(normalized)
            terms.append(term)
    return terms

category_text = product_df['categories'].fillna('').map(
    lambda value: ' '.join(unique_category_labels(value))
)
product_text = product_df[['name', 'brand']].fillna('').agg(' '.join, axis=1)
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def encode_texts(texts):
    return embedding_model.encode(
        texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )

category_matrix = encode_texts(category_text.tolist())
product_matrix = encode_texts(product_text.tolist())

review_rows = df[product_columns].copy()
review_rows['review_text'] = (
    df[['reviews.title', 'reviews.text']].fillna('').agg(' '.join, axis=1).str.strip()
)
review_rows = review_rows[review_rows['review_text'].ne('')].merge(
    product_df[product_columns].assign(_product_index=np.arange(len(product_df))),
    on=product_columns, how='left', validate='many_to_one',
)
if review_rows.empty:
    raise ValueError('No non-empty review text is available for embedding.')

review_embeddings = encode_texts(review_rows['review_text'].tolist())
review_indices = review_rows['_product_index'].to_numpy(dtype=np.int64)
review_sums = np.zeros((len(product_df), review_embeddings.shape[1]), dtype=np.float32)
np.add.at(review_sums, review_indices, review_embeddings)
review_counts = np.bincount(review_indices, minlength=len(product_df))
review_matrix = normalize(review_sums / np.maximum(review_counts, 1)[:, None])

X = normalize(np.hstack([
    category_matrix * 4.0,
    product_matrix  * 2.0,
    review_matrix,
])).astype(np.float32)

print(f'Unique product records: {len(product_df):,}')
print(f"Embedding features per text block: {category_matrix.shape[1]:,}")
print(f'Embedded review rows: {len(review_rows):,}')
print(f"Combined dense matrix: {X.shape[0]:,} products x {X.shape[1]:,} features")

## Compare 4–6 clusters

Compare KMeans cluster counts using the cosine silhouette metric on unique product profiles. Silhouette measures separation in the feature space, not whether a category name makes sense to a person, so use the product counts and examples alongside the score.

In [ ]:
candidate_results = []
candidate_models = {}

for k in (4, 5, 6):
    candidate = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = candidate.fit_predict(X)
    score = silhouette_score(X, labels, metric='cosine')
    candidate_models[k] = {
        'estimator': candidate,
        'labels': labels,
    }
    counts = pd.Series(labels).value_counts()
    candidate_results.append({
        'k': k,
        'silhouette_cosine': score,
        'smallest_product_cluster': int(counts.min()),
        'largest_product_cluster': int(counts.max()),
    })

comparison = (
    pd.DataFrame(candidate_results)
    .set_index('k')
    .sort_values('silhouette_cosine', ascending=False)
)
display(comparison.style.format({'silhouette_cosine': '{:.3f}'}))

In [ ]:
# Select k to inspect, based on the comparison above.
N_CLUSTERS = 4
selected_model = candidate_models[N_CLUSTERS]
model = selected_model['estimator']
product_df['cluster_id'] = selected_model['labels']
df = df.drop(columns='cluster_id', errors='ignore').merge(
    product_df[product_columns + ['cluster_id']],
    on=product_columns, how='left', validate='many_to_one',
)

centers = np.vstack([
    np.asarray(X[product_df['cluster_id'].to_numpy() == cluster_id].mean(axis=0)).ravel()
    for cluster_id in range(N_CLUSTERS)
])
centers = normalize(centers)
cluster_rows = []

for cluster_id in range(N_CLUSTERS):
    cluster_mask = product_df['cluster_id'].to_numpy() == cluster_id
    members = product_df.loc[cluster_mask]
    products = members['name'].dropna().value_counts().head(5)
    category_counts = (
        members['categories'].dropna().map(unique_category_labels).explode()
        .value_counts().head(8)
    )
    member_positions = np.flatnonzero(cluster_mask)
    similarities = X[member_positions] @ centers[cluster_id]
    representative_positions = member_positions[np.argsort(similarities)[-3:][::-1]]
    representative_products = product_df.iloc[representative_positions]['name'].fillna('[missing name]')
    cluster_rows.append({
        'cluster_id': cluster_id,
        'product_count': len(members),
        'top_products': ' | '.join(products.index.astype(str)),
        'representative_products': ' | '.join(representative_products.astype(str)),
        'common_categories': ', '.join(category_counts.index.astype(str)),
    })

cluster_summary = pd.DataFrame(cluster_rows).set_index('cluster_id')
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    display(cluster_summary)

## Inspect product and review examples by cluster

Inspect unique product metadata and sampled review text for each cluster. Review rows receive their product's cluster assignment after the product-level clustering. If a cluster mixes unrelated products or review topics, revisit the feature weights or compare another value of `k` before assigning names.

In [ ]:
EXAMPLES_PER_CLUSTER = 2
example_rows = []

for cluster_id, members in product_df.groupby('cluster_id', sort=True):
    example_rows.append(
        members.sample(
            n=min(EXAMPLES_PER_CLUSTER, len(members)),
            random_state=42,
        )
    )

cluster_examples = pd.concat(example_rows)
cluster_examples = cluster_examples[
    ['cluster_id', 'name', 'brand', 'categories']
].sort_values('cluster_id').reset_index(drop=True)
display(cluster_examples)

In [ ]:
EXAMPLES_PER_CLUSTER = 3
review_examples = pd.concat([
    members.sample(n=min(EXAMPLES_PER_CLUSTER, len(members)), random_state=42)
    for _, members in df.groupby('cluster_id', sort=True)
])
review_examples = review_examples[
    ['cluster_id', 'name', 'reviews.title', 'reviews.text']
].sort_values('cluster_id').reset_index(drop=True)
review_examples['reviews.text'] = review_examples['reviews.text'].str.slice(0, 300)
display(review_examples)

## Interpret and name clusters

Review the representative products and common categories before using these labels. The IDs below correspond to the fixed random seed and fitted model in this notebook; refitting with different settings can change their meaning. Names are descriptive, not ground-truth product taxonomy. Update `cluster_names` to match the newly inspected product groups.

In [ ]:
cluster_names = {
    0: 'Kindle and Fire devices with accessories',
    1: 'Echo and smart-home devices',
    2: 'Fire tablets',
    3: 'Kindle e-readers and mixed streaming devices',
}

product_df['meta_category'] = product_df['cluster_id'].map(cluster_names)
df['meta_category'] = df['cluster_id'].map(cluster_names)
if product_df['meta_category'].isna().any() or df['meta_category'].isna().any():
    raise ValueError('Every cluster must have a descriptive name.')

product_category_counts = (
    product_df.groupby('meta_category', sort=False)
    .size().rename('product_profiles')
)
review_category_counts = (
    df.groupby('meta_category', sort=False).size().rename('review_rows')
)
display(
    pd.concat([product_category_counts, review_category_counts], axis=1)
    .sort_values('product_profiles', ascending=False)
)
display(product_df[['name', 'categories']].drop_duplicates().head(10))